# Quality Gate: Validate Gold Dataset

Verifies the gold parquet file before downstream analysis and ML.

### Checks
1. No zeros remain in lifetime columns
2. No outliers (values beyond Q3 + 3×IQR per die matrix)
3. Monotonic order holds for all pieces
4. OEE values are within valid range (11–16s) where not NULL
5. Per-matrix partial time statistics match expected reference values
6. Parquet round-trip integrity

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.float_format', '{:.2f}'.format)

gold_path = Path('data/gold/pieces.parquet')
assert gold_path.exists(), f"Gold file not found: {gold_path}"

df = pd.read_parquet(gold_path)
print(f"Loaded {len(df):,} rows, {len(df.columns)} columns")
print(f"Columns: {list(df.columns)}")

Loaded 169,154 rows, 17 columns
Columns: ['timestamp', 'piece_id', 'die_matrix', 'lifetime_2nd_strike_s', 'lifetime_3rd_strike_s', 'lifetime_4th_strike_s', 'lifetime_auxiliary_press_s', 'lifetime_bath_s', 'lifetime_general_s', 'oee_cycle_time_s', 'partial_furnace_to_2nd_strike_s', 'partial_2nd_to_3rd_strike_s', 'partial_3rd_to_4th_strike_s', 'partial_4th_strike_to_auxiliary_press_s', 'partial_auxiliary_press_to_bath_s', 'after_gap', 'production_run_id']


---
## Check 1: No zero values in lifetime columns

In [2]:
LIFETIME_COLS = [
    'lifetime_2nd_strike_s', 'lifetime_3rd_strike_s',
    'lifetime_4th_strike_s', 'lifetime_auxiliary_press_s',
    'lifetime_bath_s', 'lifetime_general_s'
]

zeros_found = False
for col in LIFETIME_COLS:
    if col not in df.columns:
        continue
    n_zeros = (df[col] == 0).sum()
    if n_zeros > 0:
        print(f"FAIL: {col} has {n_zeros} zeros")
        zeros_found = True

if not zeros_found:
    print("PASS: No zero values in any lifetime column")

PASS: No zero values in any lifetime column


---
## Check 2: No outliers (Q3 + 3×IQR per die matrix)

In [3]:
outliers_found = False
for col in ['lifetime_bath_s', 'lifetime_2nd_strike_s']:
    if col not in df.columns:
        continue
    for matrix in df['die_matrix'].dropna().unique():
        values = df.loc[df['die_matrix'] == matrix, col].dropna()
        q1, q3 = values.quantile(0.25), values.quantile(0.75)
        iqr = q3 - q1
        n_out = ((values > q3 + 3*iqr) | (values < q1 - 3*iqr)).sum()
        if n_out > 0:
            print(f"FAIL: {col} matrix {matrix} has {n_out} outliers")
            outliers_found = True

if not outliers_found:
    print("PASS: No outliers found in key lifetime columns")

FAIL: lifetime_bath_s matrix 5052 has 31 outliers
FAIL: lifetime_bath_s matrix 4974 has 38 outliers
FAIL: lifetime_bath_s matrix 5091 has 95 outliers
FAIL: lifetime_bath_s matrix 5090 has 230 outliers
FAIL: lifetime_2nd_strike_s matrix 5052 has 223 outliers
FAIL: lifetime_2nd_strike_s matrix 4974 has 5 outliers
FAIL: lifetime_2nd_strike_s matrix 5091 has 100 outliers
FAIL: lifetime_2nd_strike_s matrix 5090 has 259 outliers


---
## Check 3: Monotonic order holds

In [4]:
pairs = [
    ('lifetime_2nd_strike_s', 'lifetime_3rd_strike_s'),
    ('lifetime_3rd_strike_s', 'lifetime_4th_strike_s'),
    ('lifetime_4th_strike_s', 'lifetime_auxiliary_press_s'),
    ('lifetime_auxiliary_press_s', 'lifetime_bath_s'),
]

mono_ok = True
for col_a, col_b in pairs:
    if col_a not in df.columns or col_b not in df.columns:
        continue
    both = df[col_a].notna() & df[col_b].notna()
    violations = (both & (df[col_b] <= df[col_a])).sum()
    if violations > 0:
        print(f"FAIL: {col_a} >= {col_b} in {violations} rows")
        mono_ok = False

if mono_ok:
    print("PASS: All cumulative times are monotonically increasing")

PASS: All cumulative times are monotonically increasing


---
## Check 4: OEE values within valid range (11–16s)

In [5]:
if 'oee_cycle_time_s' in df.columns:
    oee_valid = df['oee_cycle_time_s'].dropna()
    n_out_of_range = ((oee_valid < 11) | (oee_valid > 16)).sum()
    if n_out_of_range > 0:
        print(f"FAIL: {n_out_of_range} OEE values outside 11–16s range")
    else:
        print(f"PASS: All {len(oee_valid):,} non-NULL OEE values are within 11–16s")
        print(f"      OEE range: {oee_valid.min():.2f}s – {oee_valid.max():.2f}s")
        print(f"      NULL OEE:  {df['oee_cycle_time_s'].isna().sum():,} pieces")

PASS: All 131,065 non-NULL OEE values are within 11–16s
      OEE range: 11.00s – 16.00s
      NULL OEE:  38,089 pieces


---
## Check 5: Per-matrix partial time statistics

Reference values from `docs/02_ManufacturingProcess.md`:

| Segment | Expected |
|---|---|
| Furnace → 2nd strike | ~17–19s |
| 2nd → 3rd strike | ~6–7s |
| 3rd → 4th strike | ~13–14s |
| 4th strike → Aux press | ~16–17s |
| Aux press → Bath | ~1.5–2s |

In [6]:
partial_cols = [
    'partial_furnace_to_2nd_strike_s',
    'partial_2nd_to_3rd_strike_s',
    'partial_3rd_to_4th_strike_s',
    'partial_4th_strike_to_auxiliary_press_s',
    'partial_auxiliary_press_to_bath_s',
]
available = [c for c in partial_cols if c in df.columns]

print("Median partial times per die matrix (seconds):")
display(df.groupby('die_matrix')[available].median().round(2))

print("\nStddev partial times per die matrix (seconds):")
display(df.groupby('die_matrix')[available].std().round(2))

Median partial times per die matrix (seconds):


,partial_furnace_to_2nd_strike_s,partial_2nd_to_3rd_strike_s,partial_3rd_to_4th_strike_s,partial_4th_strike_to_auxiliary_press_s,partial_auxiliary_press_to_bath_s
die_matrix,,,,,
4974,17.30,6.50,13.10,17.00,1.80
5052,18.30,6.90,13.70,17.30,1.60
5090,17.70,6.80,13.80,17.70,1.60
5091,18.50,7.00,13.50,17.00,1.60



Stddev partial times per die matrix (seconds):


,partial_furnace_to_2nd_strike_s,partial_2nd_to_3rd_strike_s,partial_3rd_to_4th_strike_s,partial_4th_strike_to_auxiliary_press_s,partial_auxiliary_press_to_bath_s
die_matrix,,,,,
4974,1.58,0.27,0.31,0.89,0.05
5052,1.91,0.50,0.85,1.14,0.05
5090,2.40,0.74,1.12,1.61,0.09
5091,2.42,0.74,1.22,1.04,0.08


---
## Check 6: Parquet round-trip integrity

In [7]:
df2 = pd.read_parquet(gold_path)

assert len(df2) == len(df), f"Row count mismatch: {len(df2)} vs {len(df)}"
assert list(df2.columns) == list(df.columns), "Column mismatch"

print(f"PASS: Round-trip integrity verified")
print(f"      Rows:    {len(df2):,}")
print(f"      Columns: {len(df2.columns)}")
print(f"      File:    {gold_path}")
print(f"      Size:    {gold_path.stat().st_size / 1024:.1f} KB")

print("\n=== GOLD DATASET SUMMARY ===")
print(f"Total pieces:        {len(df2):,}")
print(f"Die matrices:        {sorted(df2['die_matrix'].dropna().unique().tolist())}")
print(f"Date range:          {df2['timestamp'].min().date()} → {df2['timestamp'].max().date()}")
print(f"Production runs:     {df2['production_run_id'].nunique():,}")
print(f"After-gap pieces:    {df2['after_gap'].sum():,}")

PASS: Round-trip integrity verified
      Rows:    169,154
      Columns: 17
      File:    data/gold/pieces.parquet
      Size:    4729.1 KB

=== GOLD DATASET SUMMARY ===
Total pieces:        169,154
Die matrices:        [4974, 5052, 5090, 5091]
Date range:          2025-11-06 → 2026-03-11
Production runs:     939
After-gap pieces:    939
